## Logistic Regression from Scratch

### Overview

Logistic Regression is a supervised machine learning algorithm primarily used for binary classification problems. Unlike Linear Regression, which predicts continuous numerical values, Logistic Regression estimates the probability that an observation belongs to a particular class.

The model uses a linear combination of input features and transforms its output through the sigmoid function, producing a probability between 0 and 1. A classification threshold is then applied to convert this probability into a binary prediction.

In this notebook, I implement Logistic Regression from scratch using NumPy, focusing on understanding the mathematical foundations and the optimization process behind the algorithm.

In [79]:
import numpy as np

In [96]:
def sigmoid_fn(z: int)-> int:
    return 1/(1+np.e**-z)
print(sigmoid_fn(-np.inf), sigmoid_fn(0), sigmoid_fn(np.inf))

0.0 0.5 1.0


In [97]:
def cost_fn(y_target, y_pred):
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean((y_target * np.log(y_pred)) + ((1 - y_pred) * (np.log(1 - y_pred))))

In [98]:
print(sigmoid_fn(0.693))
print(sigmoid_fn(0.693*2))

0.6666339590733064
0.7999528980616638


## Understanding Feature Weights

Logistic Regression is linear in log-odds, not probability. This is important because feature weights have a direct interpretation in terms of how they influence the odds of belonging to class 1.

### 1. Understanding z, probability, odds and log-odds

| z (log-odds) | Probability | Odds | Interpretation |
|---:|---:|---:|---|
| 0 | 50% | 1:1 | Both classes are equally likely |
| 0.693 | 66.7% | 2:1 | Class 1 is twice as likely |
| 1.386 | 80% | 4:1 | Class 1 is four times as likely |
| 2.197 | 90% | 9:1 | Class 1 is nine times as likely |
| -0.693 | 33.3% | 1:2 | Class 0 is twice as likely |
| -1.386 | 20% | 1:4 | Class 0 is four times as likely |

The model's linear output represents log-odds. The sigmoid function transforms this value into a probability.

### 2. How to interpret feature weights

A feature weight describes how much the log-odds change when its corresponding feature increases by one unit, assuming all other features remain constant.

To interpret the effect on odds, we use the exponential of the weight.

| Weight | Odds ratio | Interpretation |
|---:|---:|---|
| 0.693 | 2.0 | Odds double |
| 0.405 | 1.5 | Odds increase by 50% |
| 0 | 1.0 | No change in odds |
| -0.405 | 0.667 | Odds decrease by 33.3% |
| -0.693 | 0.5 | Odds are halved |

For example, imagine predicting whether a customer purchases a product based on their age.

If the learned weight for age is 0.2, every additional year multiplies the odds of purchasing by approximately 1.22, assuming all other features remain constant.

A positive weight increases the odds of class 1, while a negative weight decreases them.

In [112]:
X = np.array([[1,0],[2,1],[3,1],[4,2]], dtype=float)
y_target = np.array([0,0,1,1], dtype=float)
m, _ = X.shape
print(X)
print(y_target)

[[1. 0.]
 [2. 1.]
 [3. 1.]
 [4. 2.]]
[0. 0. 1. 1.]


In [113]:
X = np.hstack((np.ones((m, 1)), X))
m, n = X.shape
print(X)

[[1. 1. 0.]
 [1. 2. 1.]
 [1. 3. 1.]
 [1. 4. 2.]]


In [114]:
weights = np.zeros(n)
print(weights)

[0. 0. 0.]


In [115]:
print(X, weights)

[[1. 1. 0.]
 [1. 2. 1.]
 [1. 3. 1.]
 [1. 4. 2.]] [0. 0. 0.]


In [116]:
epoch = 500
lr = 0.05
cost_hist = []
weights = np.zeros(n)
for _ in range(epoch):
    z = X @ weights
    y_pred = sigmoid_fn(z)
    cost_hist.append(cost_fn(y_target, y_pred))

    for j in range(n):
        gradient = np.mean((y_pred - y_target) * X[:, j])
        weights[j] -= lr * gradient
print(weights)
print(cost_hist[0], cost_hist[-1])

[-2.47952237  0.9527032   0.4936182 ]
0.6931471805599452 0.38042033069093295


In [117]:
p_preds = sigmoid_fn(X @ weights)
p_preds[p_preds < 0.5] = 0
p_preds[p_preds >= 0.5] = 1
print(p_preds, y_target)
accuracy = sum(p_preds == y_target) / len(p_preds)
print(accuracy)

[0. 0. 1. 1.] [0. 0. 1. 1.]
1.0



### Confusion Matrix

A binary classification model predicts whether an observation belongs to class 0 (negative) or class 1 (positive).

There are four possible outcomes:

| Outcome | Actual Label | Predicted Label | Interpretation |
|:---|:---:|:---:|:---|
| **True Positive (TP)** | 1 | 1 | Correctly identifies a positive case. |
| **False Positive (FP)** | 0 | 1 | False alarm: predicts positive when the actual class is negative. |
| **True Negative (TN)** | 0 | 0 | Correctly identifies a negative case. |
| **False Negative (FN)** | 1 | 0 | Missed detection: predicts negative when the actual class is positive. |

The confusion matrix summarizes these outcomes:

|  | Predicted Positive | Predicted Negative |
|:---|:---:|:---:|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

For example, consider a medical diagnostic model that predicts whether a patient has a disease.

A False Negative means the patient actually has the disease, but the model predicts that they are healthy. This can be significantly more dangerous than a False Positive, where a healthy patient is incorrectly flagged as potentially ill.


### How to Penalize False Negatives

When False Negatives are more costly than False Positives, there are two main approaches.

- Approach 1: Adjust the Classification Threshold
- Approach 2: Increase the Positive Class Weight

Measures
- Accuracy measures the proportion of all predictions that the model classifies correctly.
- Precision measures how many of the observations predicted as positive are actually positive.
- Recall measures how many of the actual positive observations the model successfully identifies.